<!-- nav -->
[← 8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) · [Home](../../README.md) · [8.13 · SystemVerilog Types →](08_13_SystemVerilog_Types.ipynb)

# 8.12 · POSIX ERE

POSIX's **extended regular expressions** (IEEE Std 1003.1, XBD 9.4) write their operators as themselves, `(a|b)+c?`,
where a BRE escapes them ([8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb)): `|` alternation, `+` and `?` repetitions, `{1,3}`
intervals; they stand an anchor anywhere, and have no backreferences. `egrep` and `grep -E` read them, as a list of
patterns a line each, and `awk` reads them in its programs, writing their characters with C's escapes. `POSIX ERE` is
the dialect, of the POSIX family beside `POSIX BRE`, each reading the other; and C++'s `std::regex` reads its terms when
its flags name the `extended`, `awk` or `egrep` grammar ([8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb), 08.10.4).

This page builds an ERE, then what its grammar has that a BRE's has not, and what it lacks; then egrep's lists and awk's
escapes, what validation says, how C++ reads an ERE, and the editions it covers. It starts from `dialects()`; `S` is its
holder namespace, and `char`, `alternative`, `pattern` and `anchor` are the specs of the previous pages:

In [1]:
from mbse.Expressions.Cpp import RegExp as CppRegExp
from mbse.Expressions.Framework import Evaluators, Standards, Symbolics, Terms
from mbse.Expressions.POSIX import BRE, ERE
from toolkit import dialects

store = dialects()
S = store.Expressions
char = lambda c: lambda u: u.character(lambda r: r.value(ord(c)))
alternative = lambda *elements: lambda u: u.alternative(lambda r: r.elements(list(elements)))
pattern = lambda *alternatives: lambda u: u.pattern(lambda r: r.alternatives(list(alternatives)))
anchor = lambda kind: lambda u: u.assertion(lambda r: r.assertionKind(kind))
DIALECT = ERE.Expressions.DIALECT


def refused(evaluate, expression):
    """What `evaluate` says of `expression`: its value, or the reason it refuses it."""
    try:
        return evaluate(expression)
    except (NotImplementedError, ValueError) as error:
        return str(error)

<a id="T08.12.1"></a>
## 08.12.1 · An ERE, `ab|c` and `^(a|b)+c?$`

```text
ab|c        ^(a|b)+c?$
```

An ERE's kinds are a BRE's, named alike, but for its backreference, which it lacks: `ab|c` is a `pattern` of two
`alternative`s; `^(a|b)+c?$` holds anchors, a `capturinggroup` of two alternatives, and two `quantifier`s, `+` (at least
one) and `?` (at most one). `ERE.Text.ToText` writes each as an ERE does, its operators unescaped; the same tree, held by
a BRE's holder, prints as a BRE writes it:

In [2]:
choice = pattern(alternative(char("a"), char("b")), alternative(char("c")))
grouped = pattern(alternative(
    anchor("start"),
    lambda u: u.quantifier(lambda r: r.min(1).element(lambda u: u.capturinggroup(
        lambda g: g.alternatives([alternative(char("a")), alternative(char("b"))])))),
    lambda u: u.quantifier(lambda r: r.min(0).max(1).element(char("c"))),
    anchor("end")))
with store.transaction():
    eres = [S.POSIX.ERE.Expression(lambda e, spec=spec: e.root(spec)) for spec in (choice, grouped)]
    bres = [S.POSIX.BRE.Expression(lambda e, spec=spec: e.root(spec)) for spec in (choice, grouped)]
for ere, bre in zip(eres, bres):
    print(f"{ERE.Text.ToText(ere):12}", f"as a BRE: {BRE.Text.ToText(bre):26}",
          ", ".join(dict.fromkeys(Terms.kind_of(t)[0] for t in Terms.walk(ere))))
print(DIALECT.name(), DIALECT.tag, "| its kinds:", ", ".join(DIALECT.kinds()))
assert [ERE.Text.ToText(x) for x in eres] == ["ab|c", "^(a|b)+c?$"]

ab|c         as a BRE: ab\|c                      pattern, alternative, character
^(a|b)+c?$   as a BRE: ^\(a\|b\)\{1,\}c\{0,1\}$   pattern, alternative, assertion, quantifier, capturinggroup, character
POSIX ERE posix_ere | its kinds: patternlist, pattern, alternative, capturinggroup, quantifier, characterclass, characterclassrange, characterset, character, assertion, exclass, collatingelement, equivalence, error, missing


<a id="T08.12.2"></a>
## 08.12.2 · What an ERE has, and what it lacks

```text
a^$b        a+{0,1}        x*?y{1,3}?
```

An ERE's grammar lets an anchor stand anywhere, so `a^$b` is a valid ERE, which matches nothing (XBD 9.4.9); and it lets a
quantifier repeat a repeated expression, `a+{0,1}`, which the printer writes so that it reads back the same, never as
`a+?`. That would be a lazy quantifier, POSIX.1-2024's repetition modifier, which the dialect holds as a quantifier
that is not `greedy`. It has no backreference: the dialect's `Term` has no such kind, so the builders refuse to write
one into it:

In [3]:
with store.transaction():
    held = [S.POSIX.ERE.Expression(lambda e, spec=spec: e.root(pattern(alternative(*spec)))) for spec in [
        [char("a"), anchor("start"), anchor("end"), char("b")],
        [lambda u: u.quantifier(lambda r: r.min(0).max(1).element(lambda u: u.quantifier(lambda q: q.min(1).element(char("a")))))],
        [lambda u: u.quantifier(lambda r: r.min(0).greedy(False).element(char("x"))),
         lambda u: u.quantifier(lambda r: r.min(1).max(3).greedy(False).element(char("y")))]]]
for expression in held:
    print(f"{ERE.Text.ToText(expression):10}", "|", "; ".join(DIALECT.validate(expression)) or "valid")
try:
    with store.transaction():
        S.POSIX.ERE.Expression(lambda e: e.root(lambda u: u.backreference(lambda r: r.ref(1))))
except Exception:
    print("\\1 refused by the builders | a kind of POSIX ERE:", "yes" if "backreference" in DIALECT.kinds() else "no")
assert [ERE.Text.ToText(x) for x in held] == ["a^$b", "a+{0,1}", "x*?y{1,3}?"]

a^$b       | valid
a+{0,1}    | valid
x*?y{1,3}? | valid
\1 refused by the builders | a kind of POSIX ERE: no


<a id="T08.12.3"></a>
## 08.12.3 · egrep's lists, and awk's escapes

```text
grep -E -e '^a' -e 'ab|c'        awk '/\ta+/'
```

`egrep` and `grep -E` read a list of EREs, a line each, as grep reads BREs: the same kind, `patternlist`, of this
dialect. awk reads EREs too, with one difference in how they are written (XCU awk, "Regular Expressions"): a character
may be written as C writes it in a string, `\t`, `\n`, `\001`. That is a way of writing an ERE, not a grammar of its
own, so awk has no dialect: `ERE.Text.text`, given `awk`, prints a term with C's escapes, where an ERE writes a tab as
itself (`⇥` below):

In [4]:
with store.transaction():
    listed = S.POSIX.ERE.Expression(lambda e: e.root(lambda u: u.patternlist(lambda r: r.patterns([
        pattern(alternative(anchor("start"), char("a"))), choice]))))
    tabbed = S.POSIX.ERE.Expression(lambda e: e.root(pattern(alternative(
        char("\t"), lambda u: u.quantifier(lambda r: r.min(1).element(char("a")))))))
print(ERE.Text.ToText(listed))
print("as an ERE:", ERE.Text.ToText(tabbed).replace("\t", "⇥"), "| as awk writes it:", ERE.Text.text(tabbed, True))
assert ERE.Text.ToText(listed) == "^a\nab|c" and ERE.Text.text(tabbed, True) == "\\ta+"

^a
ab|c
as an ERE: ⇥a+ | as awk writes it: \ta+


<a id="T08.12.4"></a>
## 08.12.4 · Checked as its grammar has it

Validation checks what each kind's terms may be, as XBD 9.5.3's grammar has them, and its attributes' values: a
bracket expression holds no `.` as a set, since within brackets `.` is a character; an ERE has `.` alone among sets,
no `\w`; an equivalence class's name holds no `=`. An ERE binds nothing, as a BRE binds nothing:

In [5]:
with store.transaction():
    checked = {source: S.POSIX.ERE.Expression(lambda e, spec=spec: e.root(pattern(alternative(spec)))) for source, spec in [
        ("[.]", lambda u: u.characterclass(lambda c: c.elements([lambda u: u.characterset(lambda s: s.setKind("any"))]))),
        ("\\w", lambda u: u.characterset(lambda s: s.setKind("word"))),
        ("[[==e=]]", lambda u: u.characterclass(lambda c: c.elements([lambda u: u.equivalence(lambda x: x.name("=e"))]))),
        ("(a)*", lambda u: u.quantifier(lambda r: r.min(0).element(
            lambda u: u.capturinggroup(lambda g: g.alternatives([alternative(char("a"))]))))),
    ]}
for source, expression in checked.items():
    print(f"{source:8}", "|", "; ".join(DIALECT.validate(expression)) or "valid",
          "| needs:", ", ".join(sorted(Symbolics.free(expression, DIALECT))) or "nothing")
assert DIALECT.validate(checked["(a)*"]) == []

[.]      | argument 0: argument 0: a characterclass's elements cannot hold a characterset | needs: nothing
\w       | argument 0: argument 0: CharacterSet.setKind cannot be 'word' | needs: nothing
[[==e=]] | argument 0: argument 0: argument 0: an equivalence's name cannot hold '=' | needs: nothing
(a)*     | valid | needs: nothing


<a id="T08.12.5"></a>
## 08.12.5 · Read by C++, and never evaluated

```text
std::regex r("^(a|b)+c?$", std::regex::extended);        std::regex r("^a\nab|c", std::regex::egrep);
```

A C++ `regex` whose flags name `extended`, `awk` or `egrep` holds a POSIX ERE term, which `Cpp RegExp` validates as
POSIX ERE does, then checks what C++ selects of it: a list only under `egrep`, and no lazy quantifier, the repetition
modifier the POSIX editions C++ cites lack. Nothing evaluates an ERE; the framework's interpreter refuses each kind by its
dialect's name:

In [6]:
with store.transaction():
    regexes = {source: S.Cpp.RegExp.Expression(lambda e, flag=flag, spec=spec: e.root(lambda u: u.regex(
        lambda r: r.flags([flag]).pattern(lambda t: t.posix_ere(spec))))) for source, flag, spec in [
        ("extended: ^(a|b)+c?$", "extended", grouped),
        ("egrep: ^a, ab|c", "egrep", lambda u: u.patternlist(lambda r: r.patterns([
            pattern(alternative(anchor("start"), char("a"))), choice]))),
        ("awk: ^a, ab|c", "awk", lambda u: u.patternlist(lambda r: r.patterns([
            pattern(alternative(anchor("start"), char("a"))), choice]))),
        ("extended: x*?", "extended", pattern(alternative(
            lambda u: u.quantifier(lambda r: r.min(0).greedy(False).element(char("x")))))),
    ]}
for source, regex in regexes.items():
    print(f"{source:21}", "|", "; ".join(CppRegExp.Expressions.DIALECT.validate(regex)) or "valid")
interpreter = Evaluators.Interpreter(DIALECT, {}, evaluated=frozenset())  # no kind evaluated
print(refused(interpreter, Terms.term(eres[1])))
assert CppRegExp.Expressions.DIALECT.validate(regexes["extended: ^(a|b)+c?$"]) == []

extended: ^(a|b)+c?$  | valid
egrep: ^a, ab|c       | valid
awk: ^a, ab|c         | a patternlist is not in the awk grammar
extended: x*?         | a lazy quantifier is not in the extended grammar
POSIX ERE pattern is not evaluated


<a id="T08.12.6"></a>
## 08.12.6 · Its editions

`POSIX ERE` covers the same editions as `POSIX BRE`, POSIX.1's since 2001, each by its designation and its common
names, the ISO/IEC designations C++ cites among them; each edition has every one of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
ere = store.Instances.Expressions.POSIX.ERE  # the dialect, by its global name
for designation, edition in Standards.standards(ere).items():
    print(f"{designation:21}", "|", ", ".join(Standards.tags(edition)[1:]), "|", len(Standards.kinds(edition)), "kinds")
assert list(Standards.standards(ere)) == list(Standards.standards(store.Instances.Expressions.POSIX.BRE))
assert Standards.find(ere, "ISO/IEC/IEEE 9945:2009") is Standards.find(ere, "POSIX.1-2008")

IEEE Std 1003.1-2001  | POSIX.1-2001, ISO/IEC 9945:2003 | 13 kinds
IEEE Std 1003.1-2008  | POSIX.1-2008, ISO/IEC/IEEE 9945:2009 | 13 kinds
IEEE Std 1003.1-2017  | POSIX.1-2017 | 13 kinds
IEEE Std 1003.1-2024  | POSIX.1-2024 | 13 kinds


## What you learned

- **An ERE is a `POSIX ERE` term**, its kinds a BRE's but for the backreference it lacks, printed with its operators
  unescaped.
- **Its grammar is its own**: an anchor anywhere, a repeated quantifier, POSIX.1-2024's lazy quantifier held.
- **egrep's list is a kind, awk's escapes a way of printing**: neither egrep nor awk is a dialect, since POSIX defines no
  grammar of theirs.
- **C++ reads an ERE** under its `extended`, `awk` and `egrep` grammars, checking what it selects of POSIX; nothing
  evaluates one.

That ends tutorial 8's sublanguages. Next, Part II puts expressions to work as constraints, starting with case study 9.

<!-- nav -->
[← 8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) · [Home](../../README.md) · [8.13 · SystemVerilog Types →](08_13_SystemVerilog_Types.ipynb)